# Data Preprocessing: Shiller U.S. Stock Market Data

**Project:** Multi-Horizon Stock Market Return Forecasting, Factor Premiums, and Drawdown Dynamics
**Team:** Raphael Erler, Richard Zallinger, Michael Prokop

**What this notebook does in one sentence:** it takes a messy Excel file with U.S. stock market
data since 1871 and turns it into one clean table that all later notebooks use.

Steps:
1. Open the Excel file
2. Repair the date column
3. Check the data for errors
4. Don't cheat: only use information that was really known at the time
5. Remove inflation and add dividends
6. Compute the hints the model will use (features)
7. Compute the answers the model should predict (targets)
8. Save the clean table

## 1. Open the Excel file

**Where the data comes from:** Professor Robert Shiller (Yale University, Nobel Prize in economics)
collects U.S. stock market data for every month since 1871 and publishes it for free at
https://shillerdata.com/ (file `ie_data.xls`, downloaded on 2026-10-26, saved unchanged in `data/raw/`).

The file has a few title rows at the top and a note at the bottom. We skip those and only keep
the columns we need:

| Column | Simple meaning |
|---|---|
| `P` | Price of the U.S. stock market (S&P index, average of the month) |
| `D` | Dividends: money companies pay out to their shareholders each year |
| `E` | Earnings: profit of the companies over the last 12 months |
| `CPI` | Price level of everyday goods, used to measure inflation |
| `GS10` | Interest rate of a 10-year U.S. government bond, in percent |

Shiller also offers ready-made columns (e.g. inflation-adjusted prices). We do **not** use them,
because we want to compute everything ourselves and show how. We keep one of them only to check
our own result at the end of step 5.

In [1]:
import numpy as np
import pandas as pd

raw = pd.read_excel(
    "data/raw/ie_data.xls",
    sheet_name="Data",
    skiprows=8,              # skip the title rows
    header=None,
    usecols=[0, 1, 2, 3, 4, 6, 9],
    names=["date_raw", "P", "D", "E", "CPI", "GS10", "shiller_tr_price"],
    dtype={"date_raw": str},  # read the date as text, see step 2
)
print(raw.shape)
raw.tail()

(1871, 7)


,date_raw,P,D,E,CPI,GS10,shiller_tr_price
1866,2026.07,7481.339091,NaN,NaN,333.918,4.6,5.131772e+06
1867,2026.08,7711.323333,NaN,NaN,334.98,4.68,5.272759e+06
1868,2026.09,7669.414286,NaN,NaN,335.511,4.99,5.235803e+06
1869,2026.1,7722.72,NaN,NaN,335.7765,5.24,5.268026e+06
1870,NaN,Oct price is Oct 2nd close,NaN,NaN,Sept/Oct CPI estimated,Oct GS10 is Oct 2nd value,NaN


The table above still has two problems:
- The **last row** is just a note ("Oct price is Oct 2nd close"), not data. It has no date, so we remove it.
- Because that note stands in the same columns as the numbers, Python thinks `P`, `CPI` and `GS10` are **text**.
  We tell Python they are numbers.

After that, pandas shows all numbers in a column with the same number of decimals
(e.g. `7412.5515` is shown as `7412.551500`). The value itself does not change.

In [2]:
raw = raw.dropna(subset=["date_raw"])
value_cols = ["P", "D", "E", "CPI", "GS10", "shiller_tr_price"]
raw[value_cols] = raw[value_cols].astype(float)
raw.dtypes

date_raw                str
P                   float64
D                   float64
E                   float64
CPI                 float64
GS10                float64
shiller_tr_price    float64
dtype: object

## 2. Repair the date column

Shiller writes the date as a decimal number: `1871.01` means January 1871.

**The problem:** October is `1871.10`. Excel drops the zero at the end (it thinks it is unimportant)
and stores `1871.1`, which looks like January. So every October would turn into a second January.

**The fix:** we read the date as text and put the missing zero back. Example for October 2026:
1. `"2026.1"` is split at the dot into `"2026"` and `"1"`
2. the month is filled up with a zero on the right: `"1"` becomes `"10"` (`"01"` stays `"01"`)
3. we glue it together to `"2026-10-01"` and turn it into a real date

The day is always the 1st, because the data has one row per month. It just means "this month".

Afterwards we count how often each month appears: all months must appear about the same number of times.

At the end we make the date the **index**, i.e. the label of each row. Before, the rows were just numbered
(0, 1, 2, …). With the date as label we can pick a time directly (e.g. `df.loc["2008"]` for the year 2008),
and moving data by months (step 4) or checking for missing months (step 3) works along the dates.
The old text column `date_raw` is no longer needed and is removed.

In [3]:
parts = raw["date_raw"].str.split(".", expand=True)
year = parts[0]
month = parts[1].str.ljust(2, "0")   # "1" -> "10", "01" stays "01"
raw["date"] = pd.to_datetime(year + "-" + month + "-01")

# how often does each month (1-12) appear?
print(raw["date"].dt.month.value_counts().sort_index().to_dict())

df = raw.drop(columns="date_raw").set_index("date")
df.head()

{1: 156, 2: 156, 3: 156, 4: 156, 5: 156, 6: 156, 7: 156, 8: 156, 9: 156, 10: 156, 11: 155, 12: 155}


,P,D,E,CPI,GS10,shiller_tr_price
date,,,,,,
1871-01-01,4.44,0.26,0.4,12.464061,5.320000,119.611709
1871-02-01,4.50,0.26,0.4,12.844641,5.323333,118.202554
1871-03-01,4.61,0.26,0.4,13.034972,5.326667,119.884635
1871-04-01,4.74,0.26,0.4,12.559226,5.330000,128.519432
1871-05-01,4.86,0.26,0.4,12.273812,5.333333,135.438463


Every month appears 155 or 156 times, so the dates are correct now.

## 3. Check the data for errors

We check: Is any month there twice? Is any month missing? Are there empty cells?

In [4]:
print("Months that appear twice:", df.index.duplicated().sum())

expected = pd.date_range(df.index.min(), df.index.max(), freq="MS")
print("Months missing in between:", len(expected.difference(df.index)))
print("Period:", df.index.min().date(), "to", df.index.max().date(), f"({len(df)} months)")

Months that appear twice: 0
Months missing in between: 0
Period: 1871-01-01 to 2026-10-01 (1870 months)


In [5]:
print("Empty cells per column:")
print(df.isna().sum())
df[df[["D", "E"]].isna().any(axis=1)]

Empty cells per column:
P                   0
D                   4
E                   4
CPI                 0
GS10                0
shiller_tr_price    0
dtype: int64


,P,D,E,CPI,GS10,shiller_tr_price
date,,,,,,
2026-07-01,7481.339091,NaN,NaN,333.9180,4.60,5.131772e+06
2026-08-01,7711.323333,NaN,NaN,334.9800,4.68,5.272759e+06
2026-09-01,7669.414286,NaN,NaN,335.5110,4.99,5.235803e+06
2026-10-01,7722.720000,NaN,NaN,335.7765,5.24,5.268026e+06


**What we found:**
- No month appears twice and none is missing.
- Only for the **last 4 months** are dividends and earnings empty. Companies publish these numbers
  some weeks late, so they simply do not exist yet.
- The very last month (October 2026) only has the price of October 2nd, not of the whole month.

**What we do:** we remove these last months and keep only complete months.
We do **not** fill in the empty cells with guesses: that would mean inventing numbers nobody knew yet.

In [6]:
last_complete = df[["D", "E"]].dropna().index.max()
df = df.loc[:last_complete]
print("Last complete month:", last_complete.date())
print("Empty cells left:", int(df[["P", "D", "E", "CPI", "GS10"]].isna().sum().sum()))

Last complete month: 2026-06-01
Empty cells left: 0


**Do the numbers make sense?** Prices, dividends, earnings, the price level and interest rates
can never be zero or negative here. We check that and look at the smallest and largest values.

In [7]:
print("Values <= 0:", df[["P", "D", "E", "CPI", "GS10"]].le(0).sum().to_dict())
df[["P", "D", "E", "CPI", "GS10"]].describe().round(2)

Values <= 0: {'P': 0, 'D': 0, 'E': 0, 'CPI': 0, 'GS10': 0}


,P,D,E,CPI,GS10
count,1866.00,1866.00,1866.00,1866.00,1866.00
mean,474.95,8.78,21.44,69.82,4.48
std,1089.70,16.41,44.11,86.20,2.28
min,2.73,0.18,0.16,6.28,0.62
25%,8.04,0.43,0.59,10.42,3.18
50%,19.85,0.98,2.11,23.70,3.83
75%,264.30,8.60,15.88,114.25,5.05
max,7450.03,81.70,295.39,335.12,15.32


No impossible values. The numbers grow a lot over 155 years (the price goes from about 3 to 7,450),
which is normal because of economic growth and inflation.

## 4. Don't cheat: only use information that was really known at the time

Later, our model will look at a month and try to predict the future. It must only use information
that a real person **could have known in that month**. Otherwise it is like writing an exam when
you have already seen the solutions: the result looks great but is worthless.
This mistake is called **look-ahead bias**.

**Where this could happen here:** companies report their earnings several weeks after the month ends.
So in January, nobody knew the January earnings yet.

**What we do:** when we compute the hints for the model (step 6), we use earnings and dividends
from **3 months earlier**. In April, the model only sees the numbers from January.

In [8]:
LAG = 3
df["E_lag"] = df["E"].shift(LAG)
df["D_lag"] = df["D"].shift(LAG)
df[["E", "E_lag"]].head(5)

,E,E_lag
date,,
1871-01-01,0.4,NaN
1871-02-01,0.4,NaN
1871-03-01,0.4,NaN
1871-04-01,0.4,0.4
1871-05-01,0.4,0.4


The first 3 rows of `E_lag` are empty, because there is no data from before January 1871.

## 5. Remove inflation and add dividends

**Inflation:** 1 dollar in 1871 could buy much more than 1 dollar today. To compare years fairly,
we convert all prices into today's dollars using the price level (`CPI`):
`price in today's dollars = price * CPI today / CPI back then`.
Numbers in today's dollars are called **real**.

**Dividends:** owning stocks pays you in two ways: the price goes up, and the companies pay you dividends.
We assume the dividends are invested again right away. Return including dividends is called **total return**.

So for every month we compute:
**real monthly return = (price this month + one month of dividends) / price last month, corrected for inflation, − 1**

$$r_t = \frac{P_t + D_t/12}{P_{t-1}} \cdot \frac{CPI_{t-1}}{CPI_t} - 1$$

(`D` is a yearly amount, so one month is `D / 12`.)

Multiplying all monthly returns one after another gives the **total return index**: how much
1 dollar invested in 1871 would be worth, in today's money.

In [9]:
cpi_today = df["CPI"].iloc[-1]
df["P_real"] = df["P"] * cpi_today / df["CPI"]
df["E_lag_real"] = df["E_lag"] * cpi_today / df["CPI"]

df["ret_real"] = (df["P"] + df["D"] / 12) / df["P"].shift(1) * df["CPI"].shift(1) / df["CPI"] - 1
df["tr_index"] = (1 + df["ret_real"].fillna(0)).cumprod()

# check: compare with Shiller's own calculation
check = df["shiller_tr_price"] / df["shiller_tr_price"].iloc[0]
print("1 dollar from 1871 is worth today - ours:", round(df["tr_index"].iloc[-1]), " Shiller:", round(check.iloc[-1]))
print("Do the monthly changes match? Correlation:", round(df["tr_index"].pct_change().corr(check.pct_change()), 4))

1 dollar from 1871 is worth today - ours: 42720  Shiller: 42720
Do the monthly changes match? Correlation: 1.0


Our result is the same as Shiller's own calculation (correlation 1.0 means a perfect match),
so our formula is correct.

## 6. Compute the hints the model will use (features)

These are the numbers the model will look at to guess the future. Each one only uses data from
the past (see step 4).

| Feature | Simple meaning |
|---|---|
| `mom_6m`, `mom_12m` | **Momentum:** did the market go up in the last 6 / 12 months? |
| `vol_12m` | **Volatility:** how wildly did prices jump up and down in the last 12 months? |
| `pe` | **Price/earnings ratio:** how expensive are stocks compared to the companies' profit? High = expensive |
| `cape` | **CAPE:** the same, but using the average profit of the last 10 years, so one bad year does not distort it |
| `yield_gap` | **Yield gap:** are stocks or government bonds the better deal right now? |
| `drawdown` | **Drawdown:** how far below its all-time high is the market? (e.g. −0.30 = 30 % below) |

### The formulas, with a real example (June 2026)

$TR_t$ is the total return index from step 5 in month $t$, $r_t$ the real monthly return.

**Momentum:** growth of the index over the last 6 / 12 months.

$$mom\_6m_t = \frac{TR_t}{TR_{t-6}} - 1 \qquad mom\_12m_t = \frac{TR_t}{TR_{t-12}} - 1$$

$$mom\_6m = \frac{42{,}720}{40{,}265} - 1 = 0.061 \;(+6.1\,\%) \qquad mom\_12m = \frac{42{,}720}{35{,}382} - 1 = 0.207 \;(+20.7\,\%)$$

**Volatility:** standard deviation of the last 12 monthly returns. Multiplying by $\sqrt{12}$
turns the monthly value into a yearly value (the usual way to state volatility).

$$vol\_12m_t = \text{std}(r_{t-11}, \dots, r_t) \cdot \sqrt{12}$$

$$vol\_12m = 0.0265 \cdot 3.46 = 0.092 \;(9.2\,\% \text{ per year, a calm market})$$

**P/E:** price divided by the earnings of the last 12 months (3 months late, see step 4).

$$pe_t = \frac{P_t}{E_{t-3}} = \frac{7{,}450}{261.7} = 28.5$$

So investors pay 28.5 dollars for every dollar of yearly profit.

**CAPE:** real price divided by the average real earnings of the last 10 years (120 months).

$$cape_t = \frac{P^{real}_t}{\frac{1}{120}\sum_{i=0}^{119} E^{real}_{t-3-i}} = \frac{7{,}450}{181.3} = 41.1$$

The long-term average CAPE is about 18, so 41 means stocks are very expensive right now.

**Yield gap:** the earnings yield (profit per dollar invested in stocks) minus the interest of a 10-year government bond.

$$yield\_gap_t = \frac{E_{t-3}}{P_t} - \frac{GS10_t}{100} = \frac{261.7}{7{,}450} - \frac{4.47}{100} = 0.0351 - 0.0447 = -0.0096$$

Negative means: the bond pays more (4.5 %) than the companies earn per invested dollar (3.5 %).

**Drawdown:** how far the index is below the highest value it ever reached so far.

$$drawdown_t = \frac{TR_t}{\max(TR_0, \dots, TR_t)} - 1$$

In June 2026 the index is at a new high, so the drawdown is 0. Example from the financial crisis (March 2009),
the highest value so far was reached in August 2000:

$$drawdown = \frac{4{,}985}{10{,}333} - 1 = -0.518 \;(51.8\,\% \text{ below the high})$$

In [10]:
tr = df["tr_index"]
df["mom_6m"] = tr / tr.shift(6) - 1
df["mom_12m"] = tr / tr.shift(12) - 1
df["vol_12m"] = df["ret_real"].rolling(12).std() * np.sqrt(12)   # yearly value
df["pe"] = df["P"] / df["E_lag"]
df["cape"] = df["P_real"] / df["E_lag_real"].rolling(120).mean()  # 120 months = 10 years
df["yield_gap"] = df["E_lag"] / df["P"] - df["GS10"] / 100
df["drawdown"] = tr / tr.cummax() - 1

features = ["mom_6m", "mom_12m", "vol_12m", "pe", "cape", "yield_gap", "drawdown"]
df[features].describe().round(3)

,mom_6m,mom_12m,vol_12m,pe,cape,yield_gap,drawdown
count,1860.000,1854.000,1854.000,1863.000,1744.000,1863.000,1866.000
mean,0.042,0.088,0.120,16.461,17.924,0.027,-0.122
std,0.121,0.192,0.067,9.211,7.692,0.032,0.142
min,-0.401,-0.581,0.032,5.063,4.846,-0.049,-0.768
25%,-0.031,-0.033,0.083,11.790,12.139,0.007,-0.206
50%,0.046,0.088,0.105,15.271,16.636,0.027,-0.063
75%,0.111,0.204,0.135,18.896,21.679,0.042,-0.001
max,0.688,1.513,0.674,139.088,44.852,0.152,0.000


The highest `pe` is about 139 (in 2009). That is not an error: in the financial crisis company profits
almost disappeared, so price / profit became huge. This is exactly why CAPE uses 10 years of profits:
it stays calm (highest value about 45).

## 7. Compute the answers the model should predict (targets)

For every month we compute what an investor **really earned afterwards**, over the next 1, 5 and 10 years,
as a yearly average (real, with dividends). This is what the model has to guess later.

$$y_t^{(h)} = \left(\frac{TR_{t+h}}{TR_t}\right)^{12/h} - 1, \quad h = 12, 60, 120 \text{ months}$$

These values look into the future on purpose, because they are the answers. They are never used as hints.
For the most recent months the answer does not exist yet (the future has not happened), so those cells stay empty.

In [11]:
for years in [1, 5, 10]:
    h = years * 12
    df[f"fwd_ret_{years}y"] = (tr.shift(-h) / tr) ** (12 / h) - 1

targets = ["fwd_ret_1y", "fwd_ret_5y", "fwd_ret_10y"]
df[targets].describe().round(3)

,fwd_ret_1y,fwd_ret_5y,fwd_ret_10y
count,1854.000,1806.000,1746.000
mean,0.088,0.072,0.070
std,0.192,0.078,0.051
min,-0.581,-0.132,-0.059
25%,-0.033,0.021,0.038
50%,0.088,0.074,0.070
75%,0.204,0.120,0.106
max,1.513,0.333,0.200


On average, U.S. stocks earned about 7–9 % per year after inflation. Over 1 year the results jump a lot
(from −58 % to +151 %), over 10 years they are much calmer (from −6 % to +20 % per year).

## 8. Empty cells at the start and the end

Some cells are empty on purpose:
- **At the start:** e.g. CAPE needs 10 years of history, so the first 10 years have no CAPE.
- **At the end:** e.g. the 10-year answer is missing for the last 10 years.

We **keep these rows**. The 1-year model can use more rows than the 10-year model, so each later notebook
removes only the rows it cannot use.

In [12]:
df[features + targets].isna().sum().to_frame("empty cells")

,empty cells
mom_6m,6
mom_12m,12
vol_12m,12
pe,3
cape,122
yield_gap,3
drawdown,0
fwd_ret_1y,12
fwd_ret_5y,60
fwd_ret_10y,120


In [13]:
first_full = df[features].dropna().index.min().date()
print(f"All hints available from {first_full}")
for t in targets:
    s = df[t].dropna()
    print(f"{t}: {len(s)} months with a known answer, last one starts {s.index.max().date()}")

All hints available from 1881-03-01
fwd_ret_1y: 1854 months with a known answer, last one starts 2025-06-01
fwd_ret_5y: 1806 months with a known answer, last one starts 2021-06-01
fwd_ret_10y: 1746 months with a known answer, last one starts 2016-06-01


## 9. Known weaknesses of the data (bias and limitations)

- **Cheating with future information (look-ahead bias):** prevented by using earnings and dividends
  3 months late and by building every hint only from past values.
- **Monthly averages:** Shiller uses the average price of each month. A crash in the middle of a month
  looks smaller than it really was.
- **Very old data is less exact:** numbers before 1926 were reconstructed later by researchers.
  We keep them, but later compare results with and without them.
- **Only survivors in the index (survivorship bias):** the index only contains companies that were part of it
  at that time. For our question this is fine, because an investor could only buy the index as it was back then.
- **Neighbouring rows are almost the same:** the 10-year return starting in January and the one starting in
  February share 119 of 120 months. So for testing models we never split the data randomly, but always
  train on the past and test on the later future, with a gap in between.
- **No sampling:** the data is small (about 1,860 months), so we use all of it. Random sampling would also
  mix up the time order.
- **Not included:** taxes and trading fees.

## 10. Save the clean table

In [14]:
out = df[["P", "D", "E", "CPI", "GS10", "E_lag", "D_lag", "P_real", "ret_real", "tr_index"]
         + features + targets]
out.to_csv("data/shiller_clean.csv", index_label="date")
print(out.shape)
out.tail(3)

(1866, 20)


,P,D,E,CPI,GS10,E_lag,D_lag,P_real,ret_real,tr_index,mom_6m,mom_12m,vol_12m,pe,cape,yield_gap,drawdown,fwd_ret_1y,fwd_ret_5y,fwd_ret_10y
date,,,,,,,,,,,,,,,,,,,,
2026-04-01,6957.007619,80.811267,272.9447,333.020,4.32,247.663667,79.801767,6976.477714,0.037663,39931.265758,0.014554,0.263193,0.105291,28.090546,38.965658,-0.007601,-0.016507,NaN,NaN,NaN
2026-05-01,7412.551500,81.257233,284.1664,335.123,4.48,254.693333,80.083533,7386.650270,0.059761,42317.592673,0.069708,0.238207,0.093243,29.103830,41.001635,-0.010440,0.000000,NaN,NaN,NaN
2026-06-01,7450.032857,81.703200,295.3881,333.952,4.47,261.723000,80.365300,7450.032857,0.009502,42719.712978,0.060953,0.207385,0.091672,28.465335,41.083693,-0.009570,0.000000,NaN,NaN,NaN
